In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import random
from tqdm import tqdm

from src.data.data import get_ds
from src.visualizations.umap_visualizer import UMAPLatent

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout

        self.lstm = nn.LSTM(self.input_size, self.hidden_size, self.num_layers, dropout=self.dropout)


    def forward(self, x):
        """
        x: (seq_len, batch_size, input_size=1)
        """
        _, (hidden, cell) = self.lstm(x)
        # hidden: (num_layers, batch_size, hidden_size)
        # cell: (num_layers. batch_size, hidden_size)
        return hidden, cell
    
class Decoder(nn.Module):
    def __init__(self, output_size, hidden_size, num_layers, dropout):
        super().__init__()

        self.output_size = output_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout

        self.lstm = nn.LSTM(self.output_size, self.hidden_size, self.num_layers, dropout=self.dropout)
        self.fc = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, x, hidden, cell):
        """
        x: (batch_size)
        hidden: (num_layers, batch_size, hidden_size)
        cell: (num_layers, batch_size, hidden_size)
        """
        x = x.unsqueeze(0)
        # x = (1, batch_size, 1)
        output, (hidden, cell) = self.lstm(x, (hidden, cell))
        # output: (1, batch_size, hidden_size)
        # hidden: (num_layers, batch_size, hidden_size)
        # cell: (num_layers, batch_size, hidden_size)
        prediction = self.fc(output)
        # prediction: (batch_size, output_size=1)
        return prediction, hidden, cell

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, seq_len, input_size, hidden_size, num_layers, dropout, tf_ratio=1.0):
        super().__init__()
        
        self.seq_len = seq_len
        self.tf_ratio = tf_ratio

        self.encoder = Encoder(input_size, hidden_size, num_layers, dropout)
        self.decoder = Decoder(input_size, hidden_size, num_layers, dropout)

    def forward(self, x):
        """
        x: (seq_len, batch_size)
        """
        batch_size = x.shape[1]

        # outputs: (seq_len, batch_size, input_size=1)
        outputs = torch.zeros(self.seq_len, batch_size, self.decoder.output_size)

        hidden, cell = self.encoder(x)
        # hidden: (num_layers, batch_size, hidden_size)
        # cell: (num_layers, batch_size, hidden_size)
        embedding = hidden[-1, :, :]

        input = x[0, :]
        # input: (1, batch_size)
        for t in range(1, self.seq_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = (batch_size, output_size=1)
            # hidden: (num_layers, batch_size, hidden_size)
            # cell: (num_layers, batch_size, hidden_size)
            outputs[t] = output

            teacher_force = random.random() < self.tf_ratio
            input = x[t] if teacher_force else output.squeeze(0)
        return embedding, outputs



In [ ]:
class AutoencoderTrainer(nn.Module):
    def __init__(self, train_ds, test_ds, num_classes, num_variables, seq_len, input_size, hidden_size, num_layers, dropout, tf_ratio=1.0, lr1=0.01, lr2=0.01, epochs1=500, epochs2=500, m=1.0, gamma=0.999):
        super().__init__()
        
        self.train_ds = train_ds
        self.test_ds = test_ds
        self.num_classes = num_classes
        self.num_variables = num_variables
        self.seq_len = seq_len
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.tf_ratio = tf_ratio
        self.lr1 = lr1
        self.lr2 = lr2
        self.epochs1 = epochs1
        self.epochs2 = epochs2
        self.m = m
        self.gamma = gamma
        
        
        self.autoencoders = [Autoencoder(seq_len, input_size, hidden_size, num_layers, dropout, tf_ratio) for _ in range(num_variables)]
        self.contrastive_ops = [torch.optim.Adam(ae.parameters(), lr=lr1) for ae in self.autoencoders]
        self.reconstruction_ops = [torch.optim.Adam(ae.decoder.parameters(), lr=lr2) for ae in self.autoencoders]
        self.contrastive_schedulers = [torch.optim.lr_scheduler.ExponentialLR(opt, gamma) for opt in self.contrastive_ops]
        self.reconstruction_schedulers = [torch.optim.lr_scheduler.ExponentialLR(opt, gamma) for opt in self.reconstruction_ops]

        self.contrastive_loss = SiameseContrastiveLoss(m=self.m)
        self.reconstruction_loss = torch.nn.MSELoss()

        self.contrastive_losses = [[] for _ in range(self.num_variables)]
        self.reconstruction_losses = [[] for _ in range(self.num_variables)]

    def train(self):
        # train_ds is output of get_ds
        data_load = torch.utils.data.DataLoader(self.train_ds, len(self.train_ds), True)

        # First we do contrastive loss
        print("Stage 1: Contrastive Learning")
        for epoch in tqdm(range(self.epochs1)):
            for i in range(self.num_variables):
                for data_matrix, labels in data_load:
                    inp = data_matrix[:, :, i].transpose(0, 1).unsqueeze(2).float()

                    embedding, output = self.autoencoders[i](inp)
                    loss = self.contrastive_loss(embedding, labels)
                    self.contrastive_losses[i].append(float(loss))

                    self.contrastive_ops[i].zero_grad()
                    loss.backward()
                    self.contrastive_ops[i].step()
                self.contrastive_schedulers[i].step()
            
        # Next we do reconstruction loss
        print("Stage 2: Reconstruction Loss")
        for epoch in tqdm(range(self.epochs2)):
            for i in range(self.num_variables):
                for data_matrix, labels in data_load:
                    inp = data_matrix[:, :, i].transpose(0, 1).unsqueeze(2).float()

                    embedding, output = self.autoencoders[i](inp)
                    loss = self.reconstruction_loss(inp, output)
                    self.reconstruction_losses[i].append(float(loss))

                    self.reconstruction_ops[i].zero_grad()
                    loss.backward()
                    self.reconstruction_ops[i].step()
                self.reconstruction_schedulers[i].step()

        self.plot_contrastive_losses()
        self.plot_reconstruction_losses()

    def plot_contrastive_losses(self, variable=None):
        plt.figure()
        plt.title("Contrastive Losses")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")

        if variable:
            plt.plot(self.contrastive_losses[variable], label="Variable " + str(variable + 1))
        else:
            for i in range(self.num_variables):
                plt.plot(self.contrastive_losses[i], label="Variable " + str(i + 1))
        plt.legend()
        plt.show()

    def plot_reconstruction_losses(self, variable=None):
        plt.figure()
        plt.title("Reconstruction Losses")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")

        if variable:
            plt.plot(self.reconstruction_losses[variable], label="Variable " + str(variable + 1))
        else:
            for i in range(self.num_variables):
                plt.plot(self.reconstruction_losses[i], label="Variable " + str(i + 1))
        plt.legend()
        plt.show()


    def plot_one_variable_latent_space(self, variable, use_test=True):
        ds = self.train_ds
        if use_test:
            ds = self.test_ds
        visualize_moment = torch.utils.data.DataLoader(ds, len(ds), True)
        for sample in visualize_moment:
            inp, out = sample[0].detach(), sample[1].detach()
            inp = inp[:, :, variable].unsqueeze(2).transpose(0, 1).float()
            with torch.no_grad():
                hidden, cell = self.autoencoders[variable].encoder(inp)
                embeddings = hidden[-1, :, :]
                visualizer = UMAPLatent()
                visualizer.visualize(embeddings, out, self.num_classes)
        plt.title("Latent Space for Variable " + str(variable + 1))
        plt.show()

    def plot_all_latent_spaces(self, use_test=True):
        for i in range(self.num_variables):
            self.plot_one_variable_latent_space(i, use_test=use_test)

    def plot_one_variable_reconstruction_basicmotions(self, variable, use_test=True):
        ds = self.train_ds
        if use_test:
            ds = self.test_ds
        print("Reconstruction for variable " + str(variable + 1))
        # View a random train point/reconstruction from each class
        data_load = torch.utils.data.DataLoader(ds, len(ds), False)
        for data_matrix, labels in data_load:
            inp = data_matrix[:, :, variable].transpose(0, 1).unsqueeze(2).float()
            with torch.no_grad():
                _, output = self.autoencoders[variable](inp)
                print("Blue = original, Red = reconstructed")
                plt.figure()
                index = random.randint(0, 9)
                p1 = inp[:, index, :].squeeze()
                p2 = output[:, index, :]

                plt.title("Standing")
                plt.plot(p1, c='b', label="Original")
                plt.plot(p2, c='r', label="Reconstructed")
                plt.legend()

                plt.figure()
                index = random.randint(10, 19)
                p1 = inp[:, index, :].squeeze()
                p2 = output[:, index, :]

                plt.title("Running")
                plt.plot(p1, c='b', label="Original")
                plt.plot(p2, c='r', label="Reconstructed")
                plt.legend()

                plt.figure()
                index = random.randint(20, 29)
                p1 = inp[:, index, :].squeeze()
                p2 = output[:, index, :]

                plt.title("Walking")
                plt.plot(p1, c='b', label="Original")
                plt.plot(p2, c='r', label="Reconstructed")
                plt.legend()

                plt.figure()
                index = random.randint(30, 39)
                p1 = inp[:, index, :].squeeze()
                p2 = output[:, index, :]


        plt.title("Badminton")
        plt.plot(p1, c='b', label="Original")
        plt.plot(p2, c='r', label="Reconstructed")
        plt.legend()

        plt.show()

    def plot_one_variable_reconstruction_epilepsy(self, variable, use_test=True):
        ds = self.train_ds
        if use_test:
            ds = self.test_ds
        
        data_load = torch.utils.data.DataLoader(ds, len(ds), False)
        for data_matrix, labels in data_load:
            inp = data_matrix[:, :, variable].transpose(0, 1).unsqueeze(2).float()
            with torch.no_grad():
                _, output = self.autoencoders[variable](inp)
                print("Blue = original, Red = reconstructed")
                plt.figure()
                index = random.randint(0, 33)
                p1 = inp[:, index, :].squeeze()
                p2 = output[:, index, :]

                plt.title("Epilepsy")
                plt.plot(p1, c='b', label="Original")
                plt.plot(p2, c='r', label="Reconstructed")
                plt.legend()

                plt.figure()
                index = random.randint(34, 70)
                p1 = inp[:, index, :].squeeze()
                p2 = output[:, index, :]

                plt.title("Walking")
                plt.plot(p1, c='b', label="Original")
                plt.plot(p2, c='r', label="Reconstructed")
                plt.legend()

                plt.figure()
                index = random.randint(71, 106)
                p1 = inp[:, index, :].squeeze()
                p2 = output[:, index, :]

                plt.title("Running")
                plt.plot(p1, c='b', label="Original")
                plt.plot(p2, c='r', label="Reconstructed")
                plt.legend()

                plt.figure()
                index = random.randint(107, 136)
                p1 = inp[:, index, :].squeeze()
                p2 = output[:, index, :]


                plt.title("Sawing")
                plt.plot(p1, c='b', label="Original")
                plt.plot(p2, c='r', label="Reconstructed")
                plt.legend()

                plt.show()

In [ ]:
class SingleVariablePrototypesModule(nn.Module):
    def __init__(self, autoencoder, num_classes, num_prototypes, hidden_size, use_fc=False):
        super().__init__()
        self.autoencoder = autoencoder
        self.num_classes = num_classes
        self.num_prototypes  = num_prototypes
        self.hidden_size = hidden_size
        self.use_fc = use_fc

        self.prototypes = nn.Parameter(torch.zeros(num_prototypes, hidden_size))
        self.fc_layer = nn.Linear(num_prototypes, num_classes)

    def forward(self, x):
        # Get the embedding (batch_size, hidden)
        x, _ = self.autoencoder(x)
        
        x = torch.unsqueeze(x, 1).repeat_interleave(self.num_prototypes, 1)
        distances = x - self.prototype_matrix
        distances = torch.norm(distances, dim=2)
        sim = torch.pow(1 + distances, -1)
        if self.use_fc:
            distances = self.fc_layer(distances)
        return sim
    
class SingleVariablePrototypesWrapper(nn.Module):
    def __init__(self, autoencoders, num_variables, num_classes, num_prototypes, hidden_size):
        super().__init__()
        self.autoencoders = autoencoders
        self.num_variables = num_variables
        self.num_classes = num_classes
        self.num_prototypes = num_prototypes
        self.hidden_size = hidden_size

        self.single_variable_prototype_modules = []
        for i in range(num_variables):
            self.single_variable_prototype_modules.append(
                SingleVariablePrototypesModule(
                    num_classes=num_classes,
                    num_prototypes=num_prototypes,
                    hidden_size=hidden_size
                )
            )

        self.linear = nn.Linear(num_variables * num_prototypes, num_classes)

    def forward(self, x):
        raw_output = []
        for i in range(self.num_variables):
            inp = x[:, :, i].transpose(0, 1).unsqueeze(2).float()
            embeddings, _ = self.autoencoders[i](inp)
            raw_output.append(self.single_variable_prototype_modules[i](embeddings))
        
        output = self.linear(raw_output)
        return raw_output, output

    
class PrototypesTrainer(nn.Module):
    def __init__(self, train_ds, test_ds, autoencoders, num_variables, num_classes, num_prototypes, hidden_size, epochs, lr, gamma):
        super().__init__()
        self.train_ds = train_ds
        self.test_ds = test_ds
        self.autoencoders = autoencoders
        self.num_variables = num_variables
        self.num_classes = num_classes
        self.num_prototypes = num_prototypes
        self.hidden_size = hidden_size
        self.epochs = epochs
        self.lr = lr
        self.gamma = gamma

        self.wrapper = SingleVariablePrototypesWrapper(
            autoencoders=self.autoencoders,
            num_variables=self.num_variables,
            num_classes=self.num_classes,
            num_prototypes=self.num_prototypes,
            hidden_size=self.hidden_size
        )

        # Disable gradients for the autoencoders
        for autoencoder in self.autoencoders:
            for param in autoencoder.parameters():
                param.requires_grad = False

        self.opt = torch.optim.Adam(filter(lambda x: x.requires_grad, self.parameters()), lr=self.lr)
        self.classification_loss_fn = torch.nn.CrossEntropyLoss()
        self.sched = torch.optim.lr_scheduler.ExponentialLR(self.opt, self.gamma)

    def train(self):
        data_load = torch.utils.data.DataLoader(self.train_ds, len(self.train_ds), True)
        classification_losses = []
        for epoch in tqdm(range(self.epochs)):
            for data_matrix, labels in data_load:
                pred, second_degree = self.wrapper(data_matrix.float())
                classification_loss = self.classification_loss_fn(pred, labels)
                classification_losses.append(float(classification_loss))
                
                self.opt.zero_grad()
                classification_loss.backward()
                self.opt.step()
            self.sched.step()


# BasicMotions Dataset

In [ ]:
# Load in BasicMotions dataset
class_to_index={"standing":0, "running":1, "walking":2,"badminton":3}

train_ds, test_ds = get_ds("/Users/bhaveshkalisetti/Desktop/mmbs/data/basicmotions/BasicMotions_TRAIN.ts", class_to_index), get_ds("/Users/bhaveshkalisetti/Desktop/mmbs/data/basicmotions/BasicMotions_TEST.ts", class_to_index)
len(train_ds)

In [ ]:
# Load in BasicMotions autoencoder
encoding_trainer = 